In [1]:
import os, time, subprocess, pythoncom, psutil
import polars as pl
import pandas as pd
from datetime import date, datetime, timedelta
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS & LOAD
# ══════════════════════════════════════════════════════════════════════════════
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path  = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found: {test_path}")

folder_paths = {
    "parquet":    f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OUTPUT_PERFORMANCE/OUTPUT_PERFORMANCE_COMBINE/_performance_hcm.parquet",
    "dim_tables": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Python_Code/dim_tables.xlsx",
}

print("📂 Loading parquet...")
performance_hcm = pl.read_parquet(folder_paths["parquet"])
print(f"✓ {performance_hcm.shape[0]:,} rows")

_dim_Target = pl.from_pandas(
    pd.read_excel(folder_paths["dim_tables"], sheet_name="_dim_Target", engine="openpyxl")
).with_columns(pl.col("Eff_Date").cast(pl.Date))

_dim_score_card = pl.from_pandas(
    pd.read_excel(folder_paths["dim_tables"], sheet_name="_dim_score_card", engine="openpyxl")
).with_columns(pl.col("Eff_Date").cast(pl.Date))

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
OVERRIDE_DATE = None   # None = auto D-1 | "2026-04-28"

DISPLAY_NOTEBOOK     = True
SEND_EMAIL           = True
DISPLAY_MONTH_WISE   = True
DISPLAY_WEEK_WISE    = True
DISPLAY_SUP_WISE     = True
DISPLAY_DAILY_DETAIL = True   # Supervisor Wise D-1 + Daily Wise 7 ngày

EMAIL_TO = (
    "pradeep.bahadursha@concentrix.com;"
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com"
)

EMAIL_CC = (
    "gaurav.thakkar@concentrix.com;"
    "Sunny.Munjal33@concentrix.com;"
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com"
)

# EMAIL_TO = ("huuchinh.nguyen@concentrix.com")
# EMAIL_CC = ("huuchinh.nguyen@concentrix.com")

N_MONTHS     = 3
N_WEEKS      = 8
N_DAILY_DAYS = 10

_now  = datetime.now()
days_back = 3 if _now.hour < 6 else 2   # 00:00–05:59 → 3, 06:00+ → 2

if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE MODE: {OVERRIDE_DATE}")
else:
    report_date = _now - timedelta(days=days_back)

report_date_s  = report_date.strftime("%d-%b-%Y")
report_date_d  = report_date.date()
report_dt_s    = report_date.strftime("%Y-%m-%d %H:%M")   # ← không có giây

EMAIL_SUBJECT = f"VN Performance Report Daily - {report_date_s}"

# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
HDR_BG       = "#C0003C"
HDR_PERF_BG  = "#C0003C"
HDR_MONTH_BG = "#8a0020"
HDR_WEEK_BG  = "#1a6b3a"
HDR_SUP_BG   = "#0e3d7a"
HDR_DAILY_BG = "#5a3080"   # purple — phân biệt với section khác

LG_HDR  = "#1a6b3a";  LG_HDR2 = "#145230"
LG_ROW  = "#F0FAF4";  LG_SEP  = "#E8F5EC";  LG_FG  = "#1a6b3a"
NL_HDR  = "#0e3d7a";  NL_HDR2 = "#0a2d5e"
NL_ROW  = "#F0F5FF";  NL_SEP  = "#EEF3FF";  NL_FG  = "#1A56B0"
TOT_HDR = "#444444"
MET_BG  = "#D4F4E2";  MET_FG  = "#1A5C2A"
NEAR_BG = "#FFF8DB";  NEAR_FG = "#7A5200"
MISS_BG = "#FDE8EA";  MISS_FG = "#9B1C2A"
FONT    = "font-family:Arial,sans-serif;"

METRICS_ORDER = [
    ("03","#Vol"),("04","AHT"),("08","LC(%)"),("12","NPS(%)"),
    ("16","Survey"),("18","Promoter"),("20","Detractor"),("22","Neutral"),
    ("28","Exceed_Chat(%)"),("32","FCR"),("46","Re_Direct (%)"),
    ("66","Point"),("70","T3 (%)"),
]
IDX_LABEL     = dict(METRICS_ORDER)
HIGHER_BETTER = {"NPS(%)","FCR"}
LOWER_BETTER  = {"AHT","LC(%)","Exceed_Chat(%)"}

# ══════════════════════════════════════════════════════════════════════════════
# DIM TARGET
# ══════════════════════════════════════════════════════════════════════════════
def get_target(lob, metric, cur_date):
    f = (
        _dim_Target
        .filter((pl.col("LOB")==lob)&(pl.col("Metric")==metric)&(pl.col("Eff_Date")<=cur_date))
        .sort("Eff_Date", descending=True).head(1)
    )
    return (None,None) if f.is_empty() else (f["Target"][0], f["Weight"][0])

def get_tv(lob, metric, cur_date):
    t,_ = get_target(lob, metric, cur_date); return t

# ══════════════════════════════════════════════════════════════════════════════
# LOB + FILTER
# ══════════════════════════════════════════════════════════════════════════════
df = performance_hcm.filter(pl.col("Duplicate_Flag") == 0).with_columns(
    pl.when(
        pl.col("LOB").str.to_uppercase().str.contains("NON") |
        pl.col("LOB").str.to_uppercase().str.contains("NL")
    ).then(pl.lit("NL Chat")).otherwise(pl.lit("LG Chat")).alias("_safe_lob")
)

_report_month = report_date.strftime("%y_%m")
all_periods = sorted(df["_PST.Month"].drop_nulls().unique().to_list())
all_weeks   = sorted(df["_PST.Week"].drop_nulls().unique().to_list())
all_dates   = sorted(df["_PST.Date"].drop_nulls().unique().to_list())

periods = [p for p in all_periods if p <= _report_month][-N_MONTHS:]
weeks   = [w for w in all_weeks   if w <= report_date.strftime("%y_%V")][-N_WEEKS:]
daily_dates = [d for d in all_dates if d <= report_date_d][-N_DAILY_DAYS:]

print(f"✓ Periods: {periods} | Weeks: {weeks}")
print(f"✓ Daily dates: {daily_dates}")

df_m = df.filter(pl.col("_PST.Month").is_in(periods))
df_w = df.filter(pl.col("_PST.Week").is_in(weeks))
df_d1= df.filter(pl.col("_PST.Date") == report_date_d)           # D-1 only
df_d7= df.filter(pl.col("_PST.Date").is_in(daily_dates))          # 7 ngày

MAX_DATE = df["_PST.Date"].max()

TARGETS = {
    lob: {
        "NPS(%)":         get_tv(lob,"NPS",MAX_DATE),
        "FCR":            get_tv(lob,"FCR",MAX_DATE),
        "AHT":            get_tv(lob,"AHT",MAX_DATE),
        "LC(%)":         (get_tv(lob,"LC", MAX_DATE) or 0)*100,
        "Exceed_Chat(%)":(get_tv(lob,"EXC",MAX_DATE) or 0)*100,
    }
    for lob in ["LG Chat","NL Chat"]
}
print(f"✓ Periods: {periods} | Weeks: {weeks}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE
# ══════════════════════════════════════════════════════════════════════════════
def compute(frame: pl.DataFrame, lob: str) -> dict:
    null_r = {idx: None for idx,_ in METRICS_ORDER}
    if frame.is_empty(): return null_r
    cur = frame["_PST.Date"].max()
    vol = frame.select(pl.col("_conver_unique").n_unique()).item()
    if vol == 0: return null_r

    aht = frame.group_by("_conver_unique").agg(
        pl.col("Handle Time (Sum)").max().alias("ht"))["ht"].mean()

    lc_raw  = frame["_lc"].cast(pl.Float64).sum() / vol

    pr = frame["_promoter"].sum(); de = frame["_detractor"].sum()
    ne = frame["_neutral"].sum();  su = frame["_survey"].sum()
    nps_raw = (pr - de) / su if su and su > 0 else None

    # Exceed Chat: đã dedup, sum trực tiếp
    exc = frame["Exceed Chat"].cast(pl.Float64).fill_null(0).sum()

    passed = frame["Passed Sessions"].cast(pl.Float64, strict=False).fill_null(0).sum()
    totals = frame["Total Sessions"].cast(pl.Float64, strict=False).fill_null(0).sum()
    fcr_raw = passed / totals if totals > 0 else None

    # Re_Direct: đã dedup, sum trực tiếp
    rd  = frame["Re_Direct"].fill_null(0).cast(pl.Float64).sum()
    t3s = frame["T3"].cast(pl.Float64, strict=False).fill_null(0).sum()
    hcs = frame["Handle (Count)"].cast(pl.Float64, strict=False).fill_null(0).sum()

    # ── Attendance helpers ─────────────────────────────────────────────────
    def att_h(actual, target):
        if actual is None or target is None or target == 0: return None
        return min(actual / target, 1.0)

    def att_l(actual, target):
        if actual is None or target is None or actual == 0: return None
        return min(target / actual, 1.0)

    # ── Fetch targets + weights ────────────────────────────────────────────
    nps_t, nps_w = get_target(lob, "NPS", cur)
    lc_t,  lc_w  = get_target(lob, "LC",  cur)
    aht_t, aht_w = get_target(lob, "AHT", cur)
    exc_t, exc_w = get_target(lob, "EXC", cur)
    fcr_t, fcr_w = get_target(lob, "FCR", cur)

    # ── Attendance values ──────────────────────────────────────────────────
    nps_att = att_h((nps_raw * 100) if nps_raw is not None else None, nps_t)
    lc_att  = att_l(lc_raw, lc_t)
    aht_att = att_l(aht, aht_t)
    exc_att = att_l(exc / vol if vol > 0 else None, exc_t)
    fcr_att = att_h(fcr_raw, (fcr_t / 100) if fcr_t is not None else None)

    # ── Point: NPS + LC + AHT + Exceed Chat + FCR ─────────────────────────
    components = [
        (nps_att, nps_w),
        (lc_att,  lc_w),
        (aht_att, aht_w),
        (exc_att, exc_w),
        (fcr_att, fcr_w),
    ]
    valid = [(att, w) for att, w in components if att is not None and w is not None]
    pt = sum(att * w for att, w in valid) * 100 if valid else None

    return {
        "03": vol,
        "04": round(aht, 2)           if aht     is not None else None,
        "08": round(lc_raw * 100, 2),
        "12": round(nps_raw * 100, 0) if nps_raw is not None else None,
        "16": int(su) if su else None,
        "18": int(pr) if pr else None,
        "20": int(de) if de else None,
        "22": int(ne) if ne else None,
        "28": round(exc / vol * 100, 2),
        "32": round(fcr_raw * 100, 0) if fcr_raw is not None else None,
        "46": round(rd / vol * 100, 2),
        "66": round(pt, 2)            if pt      is not None else None,
        "70": round(t3s / hcs * 100, 2) if hcs > 0 else None,
    }

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def fv(idx, v):
    if v is None or (isinstance(v,float) and pd.isna(v)): return "&#8212;"
    if idx in ("03","16","18","20","22"): return f"{int(v):,}"
    if idx == "04": return f"{v:,.2f}"
    return f"{v:.2f}"

def cc(idx, v, lob):
    lbl = IDX_LABEL.get(idx,"")
    if lbl not in (HIGHER_BETTER|LOWER_BETTER) or v is None: return ""
    tar = TARGETS.get(lob,{}).get(lbl)
    if tar is None: return ""
    if lbl in HIGHER_BETTER:
        return "c-met" if v>=tar else "c-near" if v>=tar*0.9 else "c-miss"
    return "c-met" if v<=tar else "c-near" if v<=tar*1.1 else "c-miss"

def inline_color(idx, v, lob):
    cls = cc(idx,v,lob)
    if cls=="c-met":  return f"background:{MET_BG};color:{MET_FG};font-weight:bold;"
    if cls=="c-near": return f"background:{NEAR_BG};color:{NEAR_FG};font-weight:bold;"
    if cls=="c-miss": return f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;"
    return ""

def lob_colors(lob):
    if lob=="LG Chat": return LG_ROW, LG_FG, LG_SEP, LG_HDR, LG_HDR2
    if lob=="NL Chat": return NL_ROW, NL_FG, NL_SEP, NL_HDR, NL_HDR2
    return "#F5F5F5","#444","#EEEEEE",TOT_HDR,"#333"

def lgd(inline=False):
    sp = "padding:2px 8px;margin-right:8px;font-weight:bold;"
    fs = "font-family:Arial,sans-serif;font-size:11px;"
    if inline:
        return (f'<p style="{fs}margin:6px 0 16px 0;">'
                f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met</span>'
                f'<span style="{sp}background:{NEAR_BG};color:{NEAR_FG}">&#9632; Near (&ge;90%)</span>'
                f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></p>')
    return (f'<div class="lgd">'
            f'<span style="background:{MET_BG};color:{MET_FG}">&#9632; Met</span>'
            f'<span style="background:{NEAR_BG};color:{NEAR_FG}">&#9632; Near (&ge;90%)</span>'
            f'<span style="background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></div>')

# ══════════════════════════════════════════════════════════════════════════════
# SECTION HEADER HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _perf_grp_hdr(title, subtitle, color, mode="nb"):
    title_s = f"{FONT}font-size:13px;font-weight:bold;color:#fff;margin:0;"
    sub_s   = f"{FONT}font-size:10.5px;color:#ffffff;margin:3px 0 0;"
    inner   = f'<p style="{title_s}">{title}</p><p style="{sub_s}">{subtitle}</p>'
    if mode == "email":
        return (
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" '
            f'style="margin:16px 0 4px;">'
            f'<tr><td style="background:{color};padding:9px 14px;border-radius:4px;">'
            f'{inner}</td></tr></table>'
        )
    return (
        f'<div style="background:{color};padding:9px 14px;border-radius:4px;'
        f'margin:16px 0 4px;">{inner}</div>'
    )

def _perf_note(note, mode="nb"):
    s = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
         f"border-left:3px solid {HDR_BG};padding:4px 10px;"
         f"margin:0 0 6px;border-radius:0 3px 3px 0;")
    if mode == "email":
        return f'<p style="{s}">{note}</p>'
    return f'<div style="{s}">{note}</div>'

# ══════════════════════════════════════════════════════════════════════════════
# CSS
# ══════════════════════════════════════════════════════════════════════════════
CSS_NB = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.pw{{overflow-x:auto;margin-bottom:32px;border-radius:8px;border:1px solid #e0e0e0;background:#fff}}
.pw-h{{font-size:17px;font-weight:700;color:{HDR_BG};padding:13px 18px 9px;
       border-bottom:2px solid {HDR_BG};font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-family:Arial,sans-serif;font-size:11px;white-space:nowrap}}
.t thead tr.r1 th{{padding:6px 8px;font-weight:700;font-size:11px;color:#fff !important;
   border-right:1px solid rgba(255,255,255,0.2);text-align:center;
   border-bottom:2px solid rgba(255,255,255,0.3)}}
.t thead tr.r2 th{{padding:5px 8px;font-weight:600;font-size:11px;color:#fff !important;
   border-right:1px solid rgba(255,255,255,0.15);text-align:right}}
.t thead th.tl{{text-align:left !important}}
.t tbody td{{padding:4px 8px;border-bottom:1px solid #f0f0f0;text-align:right;font-size:11px}}
.t tbody td.tl{{text-align:left}}
.t tbody td.tot{{font-weight:700;background:#EFEFEF !important}}
.t tbody tr.r-lg td{{background:{LG_ROW}}}
.t tbody tr.r-nl td{{background:{NL_ROW}}}
.t tbody tr.sep-m td{{background:#1E1E2E;color:#E8E8FF;font-weight:700;
   padding:6px 14px;text-align:right;letter-spacing:0.8px;font-size:11px}}
.t tbody tr.sep-lg td{{background:{LG_SEP};color:{LG_FG};font-weight:700;
   padding:4px 12px;border-left:3px solid {LG_FG};font-size:11px}}
.t tbody tr.sep-nl td{{background:{NL_SEP};color:{NL_FG};font-weight:700;
   padding:4px 12px;border-left:3px solid {NL_FG};font-size:11px}}
.c-met{{background:{MET_BG} !important;color:{MET_FG} !important;font-weight:700}}
.c-near{{background:{NEAR_BG} !important;color:{NEAR_FG} !important;font-weight:700}}
.c-miss{{background:{MISS_BG} !important;color:{MISS_FG} !important;font-weight:700}}
.lgd{{font-family:Arial,sans-serif;font-size:11px;padding:8px 16px 12px}}
.lgd span{{padding:2px 8px;margin-right:8px;font-weight:700}}
"""

F  = "font-family:Arial,sans-serif;font-size:11px;"
TH = f"{F}padding:5px 8px;font-weight:bold;white-space:nowrap;color:#fff !important;background:{HDR_BG};"
TD = f"{F}padding:4px 8px;border:1px solid #f0f0f0;white-space:nowrap;"

def _pivot(recs, row_keys, col_key):
    return (pl.DataFrame(recs)
            .pivot(values="v", index=row_keys, on=col_key, aggregate_function="first")
            .to_pandas())

# ══════════════════════════════════════════════════════════════════════════════
# MONTH WISE
# ══════════════════════════════════════════════════════════════════════════════
def build_month_wise(mode="nb"):
    lobs  = ["LG Chat","NL Chat"]
    pcols = periods + ["Total"]
    recs  = []
    for lob in lobs:
        ldf = df_m.filter(pl.col("_safe_lob")==lob)
        for p in periods:
            m = compute(ldf.filter(pl.col("_PST.Month")==p), lob)
            for idx,lbl in METRICS_ORDER:
                recs.append({"LOB":lob,"Period":p,"index":idx,"label":lbl,"v":m[idx]})
        mt = compute(ldf, lob)
        for idx,lbl in METRICS_ORDER:
            recs.append({"LOB":lob,"Period":"Total","index":idx,"label":lbl,"v":mt[idx]})
    for p in periods:
        mg = compute(df_m.filter(pl.col("_PST.Month")==p),"LG Chat")
        for idx,lbl in METRICS_ORDER:
            recs.append({"LOB":"Total","Period":p,"index":idx,"label":lbl,"v":mg[idx]})
    mg_all = compute(df_m,"LG Chat")
    for idx,lbl in METRICS_ORDER:
        recs.append({"LOB":"Total","Period":"Total","index":idx,"label":lbl,"v":mg_all[idx]})

    piv     = _pivot(recs, ["index","label","LOB"], "Period")
    lob_hdr = [("LG Chat",LG_HDR,LG_HDR2),("NL Chat",NL_HDR,NL_HDR2),("Total",TOT_HDR,"#333")]

    header = _perf_grp_hdr(
        f"📊 Month Wise — {' | '.join(periods)}",
        f"Performance by LOB for {N_MONTHS} months. "
        f"LG Chat = Lodging &nbsp;|&nbsp; NL Chat = Non-Lodging &nbsp;|&nbsp; "
        f"Total column in bold",
        HDR_MONTH_BG, mode
    )
    note = _perf_note(
        f"Periods: {', '.join(periods)} &nbsp;|&nbsp; "
        f"&#9632; Green = Met &nbsp; &#9632; Yellow = Near (&ge;90%) &nbsp; &#9632; Red = Miss &nbsp;|&nbsp; "
        f"Targets from dim_Target (Eff_Date &le; {MAX_DATE})",
        mode
    )

    if mode=="nb":
        h = ['<div class="pw"><div style="overflow-x:auto">']
        h.append('<table class="t"><thead>')
        h.append(f'<tr class="r1" style="background:{HDR_BG}">')
        h.append(f'<th class="r1 tl" colspan="1" style="background:#8a0020">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th class="r1" colspan="{len(pcols)}" style="background:{bg}">{lob}</th>')
        h.append('</tr><tr>')
        h.append('<th class="r2 tl">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for p in pcols:
                bold = "<b>"+p+"</b>" if p=="Total" else p
                h.append(f'<th class="r2" style="background:{bg2} !important;color:#fff !important">{bold}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td class="tl">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row = sub[sub["LOB"]==lob]
                for p in pcols:
                    v = row[p].values[0] if len(row) and p in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    c  = cc(idx,v,lob) if lob!="Total" else ""
                    tc = " tot" if p=="Total" else ""
                    h.append(f'<td class="{(c+tc).strip()}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table></div>'); h.append(lgd()); h.append('</div>')
        return header + note + "".join(h)
    else:
        h = [f'<table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse;{F}mso-table-lspace:0;mso-table-rspace:0"><thead>']
        h.append('<tr>')
        h.append(f'<th style="{TH}text-align:left">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th style="{TH}text-align:center;background:{bg}" colspan="{len(pcols)}">{lob}</th>')
        h.append('</tr><tr>')
        h.append(f'<th style="{TH}text-align:left">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for p in pcols:
                bld = "font-weight:bold;" if p=="Total" else ""
                h.append(f'<th style="{TH}text-align:right;background:{bg2};{bld}">{p}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td style="{TD}">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row_bg,_fg,_sep,_hdr,_hdr2 = lob_colors(lob)
                row = sub[sub["LOB"]==lob]
                for p in pcols:
                    v = row[p].values[0] if len(row) and p in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    cs = inline_color(idx,v,lob) if lob!="Total" else ""
                    tw = "font-weight:bold;" if p=="Total" else ""
                    h.append(f'<td style="{TD}text-align:right;background:{row_bg};{cs}{tw}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table>'); h.append(lgd(inline=True))
        return header + note + "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# WEEK WISE
# ══════════════════════════════════════════════════════════════════════════════
def build_week_wise(mode="nb"):
    lobs  = ["LG Chat","NL Chat"]
    wcols = [f"Week {w}" for w in weeks] + ["Total"]
    recs  = []
    for lob in lobs:
        ldf = df_w.filter(pl.col("_safe_lob")==lob)
        for w in weeks:
            m = compute(ldf.filter(pl.col("_PST.Week")==w), lob)
            for idx,lbl in METRICS_ORDER:
                recs.append({"LOB":lob,"Week":f"Week {w}","index":idx,"label":lbl,"v":m[idx]})
        mt = compute(ldf, lob)
        for idx,lbl in METRICS_ORDER:
            recs.append({"LOB":lob,"Week":"Total","index":idx,"label":lbl,"v":mt[idx]})

    piv     = _pivot(recs, ["index","label","LOB"], "Week")
    lob_hdr = [("LG Chat",LG_HDR,LG_HDR2),("NL Chat",NL_HDR,NL_HDR2)]

    header = _perf_grp_hdr(
        f"📅 Week Wise — Last {N_WEEKS} Weeks",
        f"Weeks: {', '.join(weeks[-4:])} ... &nbsp;|&nbsp; "
        f"LG Chat & NL Chat breakdown by week &nbsp;|&nbsp; Total = cumulative",
        HDR_WEEK_BG, mode
    )
    note = _perf_note(
        f"Last {N_WEEKS} weeks shown &nbsp;|&nbsp; "
        f"Week format: YY_WW &nbsp;|&nbsp; "
        f"&#9632; Green = Met &nbsp; &#9632; Yellow = Near (&ge;90%) &nbsp; &#9632; Red = Miss",
        mode
    )

    if mode=="nb":
        h = ['<div class="pw"><div style="overflow-x:auto">']
        h.append('<table class="t"><thead>')
        h.append(f'<tr class="r1" style="background:{HDR_BG}">')
        h.append(f'<th class="r1 tl" colspan="1" style="background:#8a0020">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th class="r1" colspan="{len(wcols)}" style="background:{bg}">{lob}</th>')
        h.append('</tr><tr>')
        h.append('<th class="r2 tl">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for w in wcols:
                bold = "<b>"+w+"</b>" if w=="Total" else w
                h.append(f'<th class="r2" style="background:{bg2} !important;color:#fff !important">{bold}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td class="tl">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row = sub[sub["LOB"]==lob]
                for w in wcols:
                    v = row[w].values[0] if len(row) and w in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    c  = cc(idx,v,lob)
                    tc = " tot" if w=="Total" else ""
                    h.append(f'<td class="{(c+tc).strip()}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table></div>'); h.append(lgd()); h.append('</div>')
        return header + note + "".join(h)
    else:
        h = [f'<table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse;{F}mso-table-lspace:0;mso-table-rspace:0"><thead>']
        h.append('<tr>')
        h.append(f'<th style="{TH}text-align:left">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th style="{TH}text-align:center;background:{bg}" colspan="{len(wcols)}">{lob}</th>')
        h.append('</tr><tr>')
        h.append(f'<th style="{TH}text-align:left">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for w in wcols:
                bld = "font-weight:bold;" if w=="Total" else ""
                h.append(f'<th style="{TH}text-align:right;background:{bg2};{bld}">{w}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td style="{TD}">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row_bg = LG_ROW if lob=="LG Chat" else NL_ROW
                row = sub[sub["LOB"]==lob]
                for w in wcols:
                    v = row[w].values[0] if len(row) and w in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    cs = inline_color(idx,v,lob)
                    tw = "font-weight:bold;" if w=="Total" else ""
                    h.append(f'<td style="{TD}text-align:right;background:{row_bg};{cs}{tw}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table>'); h.append(lgd(inline=True))
        return header + note + "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# SUPERVISOR WISE (month-based, giữ nguyên)
# ══════════════════════════════════════════════════════════════════════════════
def build_sup_wise(mode="nb"):
    SUP_IDX = {
        "Volumes":"03","NPS %":"12","Surveys":"16","Promoters":"18","Neutrals":"22",
        "Detractors":"20","FCR %":"32","AHT":"04","LC %":"08",
        "Re-Direct (%)":"46","Exceed Chat (%)":"28","Point":"66","T3 (%)":"70",
    }
    LOB_MAP = {
        "NPS %":"NPS(%)","FCR %":"FCR","LC %":"LC(%)",
        "Exceed Chat (%)":"Exceed_Chat(%)","AHT":"AHT"
    }
    SCOLS = list(SUP_IDX.keys())
    sups  = sorted(df_m["Supervisor Name"].drop_nulls().unique().to_list())
    rows  = []

    for period in periods:
        pf = df_m.filter(pl.col("_PST.Month")==period)
        for lob in ["LG Chat","NL Chat"]:
            lf = pf.filter(pl.col("_safe_lob")==lob)
            for sup in sups:
                m = compute(lf.filter(pl.col("Supervisor Name")==sup), lob)
                if m["03"] is None: continue
                row = {"Month":period,"LOB":lob,"Supervisor":sup}
                for col,idx in SUP_IDX.items(): row[col] = m[idx]
                rows.append(row)

    res = pd.DataFrame(rows)

    header = _perf_grp_hdr(
        f"👥 Supervisor Wise — {' | '.join(periods)}",
        f"Individual performance per Team Leader × LOB × Period. "
        f"Sorted by Month → LOB → Supervisor Name.",
        HDR_SUP_BG, mode
    )
    note = _perf_note(
        f"Metrics: Vol, NPS%, FCR%, AHT, LC%, Exceed Chat%, Re-Direct%, Point, T3% &nbsp;|&nbsp; "
        f"Color thresholds from dim_Target (Eff_Date &le; {MAX_DATE}) &nbsp;|&nbsp; "
        f"&#9632; Green = Met &nbsp; &#9632; Yellow = Near (&ge;90%) &nbsp; &#9632; Red = Miss",
        mode
    )

    if res.empty:
        empty = '<div class="pw"><p style="padding:12px">No data</p></div>'
        return header + note + empty

    span = len(SCOLS)+3

    if mode=="nb":
        h = ['<div class="pw"><div style="overflow-x:auto">']
        h.append('<table class="t"><thead><tr>')
        for col in ["Month","LOB","Supervisor Name"]:
            h.append(f'<th class="r2 tl">{col}</th>')
        for col in SCOLS:
            h.append(f'<th class="r2">{col}</th>')
        h.append('</tr></thead><tbody>')
    else:
        h = [f'<table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse;{F}mso-table-lspace:0;mso-table-rspace:0"><thead><tr>']
        for col in ["Month","LOB","Supervisor Name"]:
            h.append(f'<th style="{TH}text-align:left">{col}</th>')
        for col in SCOLS:
            h.append(f'<th style="{TH}text-align:right">{col}</th>')
        h.append('</tr></thead><tbody>')

    prev_m = prev_lob = None
    for _, row in res.sort_values(["Month","LOB","Supervisor"]).iterrows():
        m, lob, sup            = row["Month"], row["LOB"], row["Supervisor"]
        row_bg, lfc, sep_bg, _, __ = lob_colors(lob)

        if m != prev_m:
            prev_m = m; prev_lob = None
            if mode=="nb":
                h.append(f'<tr class="sep-m"><td colspan="{span}">{m}</td></tr>')
            else:
                h.append(f'<tr><td colspan="{span}" style="{F}background:#1E1E2E;color:#E8E8FF;font-weight:bold;padding:6px 12px;text-align:right;letter-spacing:0.8px">{m}</td></tr>')

        if lob != prev_lob:
            prev_lob = lob
            if mode=="nb":
                cls = "sep-lg" if lob=="LG Chat" else "sep-nl"
                h.append(f'<tr class="{cls}"><td colspan="{span}">{lob}</td></tr>')
            else:
                h.append(f'<tr><td colspan="{span}" style="{F}background:{sep_bg};color:{lfc};font-weight:bold;padding:4px 12px;border-left:3px solid {lfc}">{lob}</td></tr>')

        if mode=="nb":
            tr = "r-lg" if lob=="LG Chat" else "r-nl"
            h.append(f'<tr class="{tr}">')
            h.append(f'<td class="tl" style="color:#000;font-size:11px;font-weight:700">{m}</td>')
            h.append(f'<td class="tl" style="color:{lfc};font-weight:700">{lob}</td>')
            h.append(f'<td class="tl">{sup}</td>')
        else:
            h.append('<tr>')
            h.append(f'<td style="{TD}background:{row_bg};color:#000;font-size:11px;font-weight:bold">{m}</td>')
            h.append(f'<td style="{TD}background:{row_bg};color:{lfc};font-weight:bold">{lob}</td>')
            h.append(f'<td style="{TD}background:{row_bg}">{sup}</td>')

        for col, idx in SUP_IDX.items():
            v   = row.get(col)
            lbl = LOB_MAP.get(col,"")
            tar = TARGETS.get(lob,{}).get(lbl)
            col_cls = ""
            if tar is not None and v is not None and not (isinstance(v,float) and pd.isna(v)):
                if lbl in HIGHER_BETTER:
                    col_cls = "c-met" if v>=tar else "c-near" if v>=tar*0.9 else "c-miss"
                elif lbl in LOWER_BETTER:
                    col_cls = "c-met" if v<=tar else "c-near" if v<=tar*1.1 else "c-miss"
            if mode=="nb":
                h.append(f'<td class="{col_cls}">{fv(idx,v)}</td>')
            else:
                cs = inline_color(idx,v,lob) if col_cls else ""
                h.append(f'<td style="{TD}text-align:right;background:{row_bg};{cs}">{fv(idx,v)}</td>')
        h.append('</tr>')

    if mode=="nb":
        h.append('</tbody></table></div>'); h.append(lgd()); h.append('</div>')
    else:
        h.append('</tbody></table>'); h.append(lgd(inline=True))

    return header + note + "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# SUPERVISOR WISE D-1  (chỉ ngày report_date)
# ══════════════════════════════════════════════════════════════════════════════
def build_sup_wise_d1(mode="nb"):
    SUP_IDX = {
        "Volumes":"03","NPS %":"12","Surveys":"16","Promoters":"18","Neutrals":"22",
        "Detractors":"20","FCR %":"32","AHT":"04","LC %":"08",
        "Re-Direct (%)":"46","Exceed Chat (%)":"28","Point":"66","T3 (%)":"70",
    }
    LOB_MAP = {
        "NPS %":"NPS(%)","FCR %":"FCR","LC %":"LC(%)",
        "Exceed Chat (%)":"Exceed_Chat(%)","AHT":"AHT"
    }
    SCOLS = list(SUP_IDX.keys())
    sups  = sorted(df_d1["Supervisor Name"].drop_nulls().unique().to_list())
    rows  = []

    for lob in ["LG Chat","NL Chat"]:
        lf = df_d1.filter(pl.col("_safe_lob")==lob)
        for sup in sups:
            m = compute(lf.filter(pl.col("Supervisor Name")==sup), lob)
            if m["03"] is None: continue
            row = {"Date": str(report_date_d), "LOB": lob, "Supervisor": sup}
            for col, idx in SUP_IDX.items(): row[col] = m[idx]
            rows.append(row)

    res = pd.DataFrame(rows)

    header = _perf_grp_hdr(
        f"👥 Supervisor Wise — D-1 ({report_date_d})",
        f"Individual performance per Team Leader × LOB for {report_date_d} only. "
        f"Sorted by LOB → Supervisor Name.",
        HDR_DAILY_BG, mode
    )
    note = _perf_note(
        f"Date: {report_date_d} &nbsp;|&nbsp; "
        f"Metrics: Vol, NPS%, FCR%, AHT, LC%, Exceed Chat%, Re-Direct%, Point, T3% &nbsp;|&nbsp; "
        f"&#9632; Green = Met &nbsp; &#9632; Yellow = Near (&ge;90%) &nbsp; &#9632; Red = Miss",
        mode
    )

    if res.empty:
        msg = f'<p style="padding:12px;color:#888;">No data for {report_date_d}</p>'
        if mode=="nb": return header + note + f'<div class="pw">{msg}</div>'
        return header + note + msg

    span = len(SCOLS) + 3

    if mode=="nb":
        h = ['<div class="pw"><div style="overflow-x:auto">']
        h.append('<table class="t"><thead><tr>')
        for col in ["Date","LOB","Supervisor Name"]:
            h.append(f'<th class="r2 tl">{col}</th>')
        for col in SCOLS:
            h.append(f'<th class="r2">{col}</th>')
        h.append('</tr></thead><tbody>')
    else:
        h = [f'<table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse;{F}mso-table-lspace:0;mso-table-rspace:0"><thead><tr>']
        for col in ["Date","LOB","Supervisor Name"]:
            h.append(f'<th style="{TH}text-align:left;background:#8a0020">{col}</th>')
        for col in SCOLS:
            h.append(f'<th style="{TH}text-align:right;background:{HDR_DAILY_BG}">{col}</th>')
        h.append('</tr></thead><tbody>')

    prev_lob = None
    for _, row in res.sort_values(["LOB","Supervisor"]).iterrows():
        lob, sup = row["LOB"], row["Supervisor"]
        row_bg, lfc, sep_bg, _, __ = lob_colors(lob)

        if lob != prev_lob:
            prev_lob = lob
            if mode=="nb":
                cls = "sep-lg" if lob=="LG Chat" else "sep-nl"
                h.append(f'<tr class="{cls}"><td colspan="{span}">{lob}</td></tr>')
            else:
                h.append(f'<tr><td colspan="{span}" style="{F}background:{sep_bg};color:{lfc};font-weight:bold;padding:4px 12px;border-left:3px solid {lfc}">{lob}</td></tr>')

        if mode=="nb":
            tr = "r-lg" if lob=="LG Chat" else "r-nl"
            h.append(f'<tr class="{tr}">')
            h.append(f'<td class="tl">{row["Date"]}</td>')
            h.append(f'<td class="tl" style="color:{lfc};font-weight:700">{lob}</td>')
            h.append(f'<td class="tl">{sup}</td>')
        else:
            h.append('<tr>')
            h.append(f'<td style="{TD}background:{row_bg}">{row["Date"]}</td>')
            h.append(f'<td style="{TD}background:{row_bg};color:{lfc};font-weight:bold">{lob}</td>')
            h.append(f'<td style="{TD}background:{row_bg}">{sup}</td>')

        for col, idx in SUP_IDX.items():
            v   = row.get(col)
            lbl = LOB_MAP.get(col,"")
            tar = TARGETS.get(lob,{}).get(lbl)
            col_cls = ""
            if tar is not None and v is not None and not (isinstance(v,float) and pd.isna(v)):
                if lbl in HIGHER_BETTER:
                    col_cls = "c-met" if v>=tar else "c-near" if v>=tar*0.9 else "c-miss"
                elif lbl in LOWER_BETTER:
                    col_cls = "c-met" if v<=tar else "c-near" if v<=tar*1.1 else "c-miss"
            if mode=="nb":
                h.append(f'<td class="{col_cls}">{fv(idx,v)}</td>')
            else:
                cs = inline_color(idx,v,lob) if col_cls else ""
                h.append(f'<td style="{TD}text-align:right;background:{row_bg};{cs}">{fv(idx,v)}</td>')
        h.append('</tr>')

    if mode=="nb":
        h.append('</tbody></table></div>'); h.append(lgd()); h.append('</div>')
    else:
        h.append('</tbody></table>'); h.append(lgd(inline=True))

    return header + note + "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# DAILY WISE — 7 ngày gần nhất
# ══════════════════════════════════════════════════════════════════════════════
def build_daily_wise(mode="nb"):
    lobs  = ["LG Chat","NL Chat"]
    dcols = [str(d) for d in daily_dates] + ["Total"]
    recs  = []

    for lob in lobs:
        ldf = df_d7.filter(pl.col("_safe_lob")==lob)
        for d in daily_dates:
            m = compute(ldf.filter(pl.col("_PST.Date")==d), lob)
            for idx,lbl in METRICS_ORDER:
                recs.append({"LOB":lob,"Date":str(d),"index":idx,"label":lbl,"v":m[idx]})
        mt = compute(ldf, lob)
        for idx,lbl in METRICS_ORDER:
            recs.append({"LOB":lob,"Date":"Total","index":idx,"label":lbl,"v":mt[idx]})

    piv     = _pivot(recs, ["index","label","LOB"], "Date")
    lob_hdr = [("LG Chat",LG_HDR,LG_HDR2),("NL Chat",NL_HDR,NL_HDR2)]

    date_labels = [str(d) for d in daily_dates]

    header = _perf_grp_hdr(
        f"📆 Daily Wise — Last {N_DAILY_DAYS} Days",
        f"Daily performance: {date_labels[0]} → {date_labels[-1]} &nbsp;|&nbsp; "
        f"LG Chat & NL Chat breakdown &nbsp;|&nbsp; Total = cumulative",
        HDR_DAILY_BG, mode
    )
    note = _perf_note(
        f"Last {N_DAILY_DAYS} days shown &nbsp;|&nbsp; "
        f"&#9632; Green = Met &nbsp; &#9632; Yellow = Near (&ge;90%) &nbsp; &#9632; Red = Miss",
        mode
    )

    if mode=="nb":
        h = ['<div class="pw"><div style="overflow-x:auto">']
        h.append('<table class="t"><thead>')
        h.append(f'<tr class="r1" style="background:{HDR_BG}">')
        h.append(f'<th class="r1 tl" colspan="1" style="background:#8a0020">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th class="r1" colspan="{len(dcols)}" style="background:{bg}">{lob}</th>')
        h.append('</tr><tr>')
        h.append('<th class="r2 tl">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for d in dcols:
                bold = "<b>Total</b>" if d=="Total" else d
                h.append(f'<th class="r2" style="background:{bg2} !important;color:#fff !important">{bold}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td class="tl">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row = sub[sub["LOB"]==lob]
                for d in dcols:
                    v = row[d].values[0] if len(row) and d in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    c  = cc(idx,v,lob)
                    tc = " tot" if d=="Total" else ""
                    h.append(f'<td class="{(c+tc).strip()}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table></div>'); h.append(lgd()); h.append('</div>')
        return header + note + "".join(h)
    else:
        h = [f'<table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse;{F}mso-table-lspace:0;mso-table-rspace:0"><thead>']
        h.append('<tr>')
        h.append(f'<th style="{TH}text-align:left;background:#8a0020">LOB</th>')
        for lob,bg,_ in lob_hdr:
            h.append(f'<th style="{TH}text-align:center;background:{bg}" colspan="{len(dcols)}">{lob}</th>')
        h.append('</tr><tr>')
        h.append(f'<th style="{TH}text-align:left;background:#8a0020">Metrics</th>')
        for _,bg,bg2 in lob_hdr:
            for d in dcols:
                bld = "font-weight:bold;" if d=="Total" else ""
                h.append(f'<th style="{TH}text-align:right;background:{bg2};{bld}">{d}</th>')
        h.append('</tr></thead><tbody>')
        for idx,lbl in METRICS_ORDER:
            sub = piv[piv["index"]==idx]
            h.append('<tr>')
            h.append(f'<td style="{TD}text-align:left">{lbl}</td>')
            for lob,_,__ in lob_hdr:
                row_bg = LG_ROW if lob=="LG Chat" else NL_ROW
                row = sub[sub["LOB"]==lob]
                for d in dcols:
                    v = row[d].values[0] if len(row) and d in row.columns else None
                    if isinstance(v,float) and pd.isna(v): v=None
                    cs = inline_color(idx,v,lob)
                    tw = "font-weight:bold;" if d=="Total" else ""
                    h.append(f'<td style="{TD}text-align:right;background:{row_bg};{cs}{tw}">{fv(idx,v)}</td>')
            h.append('</tr>')
        h.append('</tbody></table>'); h.append(lgd(inline=True))
        return header + note + "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def build_greeting():
    EXCEL_URL = "https://cnxmail.sharepoint.com/:x:/s/WFM-Expedia-HCM/IQASLnLKm4WERZIIF3Ljh5ljAfe_DaXp7L1bnvOHWKVhcfI?e=QhacSh"
    PBI_URL   = "https://app.powerbi.com/view?r=eyJrIjoiOTVmMDY5MDItZTc1YS00ZjcyLThlOTctNTZmNmI4NTM4YzgwIiwidCI6IjU5OWU1MWQ2LTJmOGMtNDM0Ny04ZTU5LTFmNzk1YTUxYTk4YyIsImMiOjZ9&pageName=d135"
    btn = (
        "display:inline-block;padding:6px 14px;border-radius:4px;"
        "font-weight:bold;text-decoration:none;font-size:11px;"
        f"font-family:Arial,sans-serif;margin-right:10px;"
    )
    return f"""
<p style="{F}font-size:12px;margin:0 0 14px 0;line-height:1.7">Hi Team,</p>
<p style="{F}font-size:12px;margin:0 0 14px 0;line-height:1.7">
    Please find below the <strong>VN Performance Report Daily - {report_dt_s}</strong>
    for period(s): <strong>{", ".join(periods)}</strong>.
</p>
<p style="{F}font-size:12px;margin:0 0 10px 0;">
    You can also access the full report via the links below:
</p>
<p style="margin:0 0 20px 0;">
    <a href="{EXCEL_URL}" target="_blank"
       style="{btn}background:#217346;color:#ffffff;border:1px solid #145230;">
       &#128202; Excel Report
    </a>
    <a href="{PBI_URL}" target="_blank"
       style="{btn}background:#F2C811;color:#000000;border:1px solid #c9a800;">
       &#128269; Power BI Dashboard
    </a>
</p>
<ul style="{F}font-size:12px;margin:0 0 16px 0;padding-left:20px;line-height:1.8">
    <li><span style="background:{MET_BG};color:{MET_FG};padding:1px 6px;font-weight:bold">Green</span> — Met target.</li>
    <li><span style="background:{NEAR_BG};color:{NEAR_FG};padding:1px 6px;font-weight:bold">Yellow</span> — Near target (&ge;90%), requires attention.</li>
    <li><span style="background:{MISS_BG};color:{MISS_FG};padding:1px 6px;font-weight:bold">Red</span> — Missed target, action needed.</li>
</ul>
<p style="{F}font-size:12px;margin:0 0 20px 0;line-height:1.7">Please review and reach out if you have any questions.</p>
"""

def build_signature():
    return f"""
<br>
<p style="{F}font-size:12px;margin:0 0 4px 0;line-height:1.7">Thanks &amp; Regards,</p>
<p style="{F}font-size:12px;font-weight:bold;margin:0 0 2px 0;">Chinh Nguyen</p>
<p style="{F}font-size:11px;color:#555;margin:0 0 2px 0;">Analyst, WFM Real Time Management</p>
<br>
<p style="{F}font-size:11px;color:#555;margin:0 0 2px 0;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{F}font-size:11px;color:#555;margin:0 0 2px 0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
              style="color:{HDR_BG};text-decoration:none;font-weight:bold;">
        huuchinh.nguyen@concentrix.com
    </a>
</p>
<p style="{F}font-size:10px;color:#aaa;margin-top:12px;
   border-top:1px solid #eeeeee;padding-top:8px;">
    Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Periods: {", ".join(periods)} &nbsp;|&nbsp;
    Source: _performance_hcm.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# BUILD & RENDER
# ══════════════════════════════════════════════════════════════════════════════
nb_parts    = []
email_parts = []

# ── Report banner ─────────────────────────────────────────────────────────────
for parts, m in [(nb_parts,"nb"),(email_parts,"email")]:
    parts.append(_perf_grp_hdr(
        f"📈 VN Performance Report Daily — {report_dt_s}",   # ← không có giây
        f"Periods: {', '.join(periods)} &nbsp;|&nbsp; "
        f"Weeks: last {N_WEEKS} &nbsp;|&nbsp; "
        f"Source: _performance_hcm.parquet",
        HDR_PERF_BG, m
    ))

if DISPLAY_MONTH_WISE:
    print("⏳ Month Wise...")
    nb_parts.append(build_month_wise(mode="nb"))
    if SEND_EMAIL: email_parts.append(build_month_wise(mode="email"))
    print("✓")

if DISPLAY_WEEK_WISE:
    print("⏳ Week Wise...")
    nb_parts.append(build_week_wise(mode="nb"))
    if SEND_EMAIL: email_parts.append(build_week_wise(mode="email"))
    print("✓")

if DISPLAY_SUP_WISE:
    print("⏳ Supervisor Wise (month)...")
    nb_parts.append(build_sup_wise(mode="nb"))
    if SEND_EMAIL: email_parts.append(build_sup_wise(mode="email"))
    print("✓")

if DISPLAY_DAILY_DETAIL:
    print("⏳ Supervisor Wise D-1...")
    nb_parts.append(build_sup_wise_d1(mode="nb"))
    if SEND_EMAIL: email_parts.append(build_sup_wise_d1(mode="email"))
    print("✓")

    print("⏳ Daily Wise (7 days)...")
    nb_parts.append(build_daily_wise(mode="nb"))
    if SEND_EMAIL: email_parts.append(build_daily_wise(mode="email"))
    print("✓")

# ── Display notebook ──────────────────────────────────────────────────────────
if DISPLAY_NOTEBOOK and nb_parts:
    nb_html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{CSS_NB}</style></head><body>"
        + "".join(nb_parts)
        + "</body></html>"
    )
    escaped = (nb_html
               .replace("&", "&amp;")
               .replace('"', "&quot;")
               .replace("'", "&#39;"))
    display(HTML(
        f'<iframe srcdoc="{escaped}" '
        f'style="width:100%;border:none;min-height:900px;" '
        f'onload="this.style.height='
        f"(this.contentDocument.body.scrollHeight+40)+'px'\""
        f'></iframe>'
    ))
    print("✓ Display done")

# ── Send email ────────────────────────────────────────────────────────────────
if SEND_EMAIL and email_parts:
    import win32com.client

    full_email = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{F}'>"
        + build_greeting()
        + "".join(email_parts)
        + build_signature()
        + "</div>"
    )

    def send_auto(to, cc, subject, html_body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower()=="outlook.exe" for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol = win32com.client.Dispatch("Outlook.Application")
            ol.GetNamespace("MAPI").Logon()
            mail = ol.CreateItem(0)
            mail.To=to; mail.CC=cc; mail.Subject=subject; mail.HTMLBody=html_body
            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, full_email, quit_after=True)

📂 Loading parquet...
✓ 1,283,864 rows
✓ Periods: ['26_07', '26_08', '26_09'] | Weeks: ['26_29', '26_30', '26_31', '26_32', '26_33', '26_34', '26_35', '26_36']
✓ Daily dates: [datetime.date(2026, 8, 24), datetime.date(2026, 8, 25), datetime.date(2026, 8, 26), datetime.date(2026, 8, 27), datetime.date(2026, 8, 28), datetime.date(2026, 8, 29), datetime.date(2026, 8, 30), datetime.date(2026, 8, 31), datetime.date(2026, 9, 1), datetime.date(2026, 9, 2)]
✓ Periods: ['26_07', '26_08', '26_09'] | Weeks: ['26_29', '26_30', '26_31', '26_32', '26_33', '26_34', '26_35', '26_36']
⏳ Month Wise...
✓
⏳ Week Wise...
✓
⏳ Supervisor Wise (month)...
✓
⏳ Supervisor Wise D-1...
✓
⏳ Daily Wise (7 days)...
✓


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


✓ Display done
✓ Sent → pradeep.bahadursha@concentrix.com;puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com;ExpediaVN_QA_Team@concentrix.com;ExpediaVN_Training_Team@concentrix.com
